# Exception Handling

**Session 06 (2026-10-24) · Priority 1 · Notebook 1 of 6**

Every program meets bad input, missing files and unexpected values. Exception handling lets your program **react** to these problems instead of crashing.

What you'll learn:

1. What an exception is, and what Python does behind the scenes
2. `try` / `except`
3. Catching the right exception
4. `else` and `finally`
5. `raise`: reporting errors from your own code
6. How exceptions travel up through function calls
7. Custom exceptions
8. Best practices
9. Practice

> Run the cells from top to bottom (`Shift + Enter`).

---
## 1. What is an exception?

An **exception** is an error that happens **while the program is running**. It's different from a **syntax error**, which Python finds before running anything.

| | Syntax error | Exception |
|---|---|---|
| When | before the code runs, while Python reads it | while the code runs |
| Example | `print("hi"` (missing bracket) | `int("abc")`, `10 / 0` |
| Can `try` / `except` catch it? | No | Yes |

Common built-in exceptions:

| Exception | Raised when | Example |
|---|---|---|
| `ValueError` | the type is right but the value is wrong | `int("abc")` |
| `TypeError` | the type is wrong for the operation | `"a" + 1` |
| `ZeroDivisionError` | dividing by zero | `10 / 0` |
| `KeyError` | a dictionary key doesn't exist | `{}["x"]` |
| `IndexError` | a list index is out of range | `[1, 2][5]` |
| `NameError` | a variable doesn't exist | `print(undefined_name)` |
| `AttributeError` | an object has no such attribute or method | `(5).append(1)` |
| `FileNotFoundError` | a file doesn't exist | `open("missing.txt")` |

### Behind the scenes

When something goes wrong, Python:

1. **Creates an exception object** that describes the problem: its type (such as `ValueError`) and a message.
2. **Stops** the current line. Nothing after it in that block runs.
3. **Looks for a handler.** It checks whether the line is inside a `try` block with a matching `except`. If not, it goes back to the function that **called** the current function and checks there, and so on up the chain of calls (the **call stack**).
4. If it finds a matching `except`, that code runs and the program **continues** after the `try` statement.
5. If it reaches the top without finding one, Python prints a **traceback** and the program **stops**.

This is what an unhandled exception looks like:

```
Traceback (most recent call last):
  File "example.py", line 2, in <module>
    number = int("abc")
ValueError: invalid literal for int() with base 10: 'abc'
```

**Read a traceback from the bottom up.** The last line gives the exception type and message. The lines above show where it happened (file, line number, the code).

Try it yourself: type `int("abc")` in a new cell and run it.

---
## 2. `try` / `except`

**Syntax**

```python
try:
    # code that might fail
except SomeError:
    # runs only if SomeError happened inside the try block
```

**How it flows**

- No error in `try`: the `except` block is skipped.
- An error in `try`: the rest of the `try` block is skipped, and the matching `except` block runs.
- Either way, the program carries on after the whole statement.

In [ ]:
print("before")
try:
    number = int("42")            # works
    print("converted:", number)
except ValueError:
    print("that was not a number")
print("after")

In [ ]:
print("before")
try:
    number = int("abc")           # fails here...
    print("converted:", number)   # ...so this line is skipped
except ValueError:
    print("that was not a number")
print("after")                    # the program keeps going

A common pattern is to wrap risky code in a small function, so the rest of your program gets a clean answer:

In [ ]:
def to_int(text):
    try:
        return int(text)
    except ValueError:
        return None

print(to_int("10"))
print(to_int("ten"))

---
## 3. Catching the right exception

### 3.1 Be specific

An `except` block catches **only** the exception types you name. Anything else still stops the program. Catch the errors you **expect** and know how to handle.

In [ ]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Cannot divide by zero")
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))

### 3.2 Several `except` blocks

Python checks the `except` blocks from top to bottom and runs the **first** one that matches. Different errors can get different messages.

In [ ]:
def parse_and_divide(a_text, b_text):
    try:
        a = int(a_text)
        b = int(b_text)
        return a / b
    except ValueError:
        print(f"Not a number: '{a_text}' or '{b_text}'")
    except ZeroDivisionError:
        print("The second number cannot be zero")

print(parse_and_divide("10", "2"))
print(parse_and_divide("ten", "2"))    # ValueError branch, then the function returns None
print(parse_and_divide("10", "0"))     # ZeroDivisionError branch

### 3.3 One block for several exceptions

If two errors need the same handling, put them in a **tuple**:

In [ ]:
def get_item(data, key):
    try:
        return data[key]
    except (KeyError, IndexError):
        return "not found"

print(get_item({"a": 1}, "a"))
print(get_item({"a": 1}, "b"))     # KeyError
print(get_item([10, 20], 5))       # IndexError

### 3.4 The exception object: `as e`

`except SomeError as e` stores the exception object in `e`:

- `print(e)` or `str(e)` gives the **message**
- `type(e).__name__` gives the **type name**

In [ ]:
try:
    int("abc")
except ValueError as e:
    print("Type   :", type(e).__name__)
    print("Message:", e)

### 3.5 The exception family tree

Exceptions are organised in a hierarchy. Catching a **parent** also catches all of its **children**.

```
BaseException
 ├── KeyboardInterrupt       (Ctrl + C)
 ├── SystemExit              (exit())
 └── Exception               ← the widest you should normally catch
      ├── ArithmeticError
      │    └── ZeroDivisionError
      ├── LookupError
      │    ├── IndexError
      │    └── KeyError
      ├── OSError
      │    ├── FileNotFoundError
      │    ├── FileExistsError
      │    └── PermissionError
      ├── TypeError
      └── ValueError
```

In [ ]:
try:
    [1, 2][9]
except LookupError as e:           # the parent of IndexError and KeyError
    print("caught by the parent class:", type(e).__name__)

### 3.6 Catching everything, and why to be careful

`except Exception` catches almost every error. That's useful at the very top of a program, to log anything unexpected:

In [ ]:
def risky(data):
    try:
        return data["total"] / data["count"]
    except Exception as e:
        print(f"Something went wrong -> {type(e).__name__}: {e}")

risky({"total": 10, "count": 0})   # ZeroDivisionError
risky({"total": 10})               # KeyError
risky(None)                        # TypeError

Inside normal code, though, catching everything **hides bugs**. Here a typo raises a `NameError`, but the broad `except` swallows it and returns a wrong answer with no hint why:

In [ ]:
def buggy_average(numbers):
    try:
        return sum(numbrs) / len(numbers)   # typo: numbrs instead of numbers
    except Exception:
        return 0                            # the typo is silently hidden!

print(buggy_average([1, 2, 3]))   # prints 0 instead of 2.0

**Never use a bare `except:`** (with no type). It catches even `KeyboardInterrupt`, so pressing Ctrl + C can't stop your program.

---
## 4. `else` and `finally`

The full form of the statement:

```python
try:
    # code that might fail
except SomeError:
    # runs if SomeError happened
else:
    # runs only if NO exception happened
finally:
    # ALWAYS runs, error or not, even after a return
```

| Block | Runs when |
|---|---|
| `try` | always, first |
| `except` | an exception happened and matched |
| `else` | the `try` block finished with no exception |
| `finally` | always, last |

**Why use `else`?** It keeps the `try` block small. Only the line that can fail goes in `try`, and the follow-up code goes in `else`, so you don't accidentally catch errors from the follow-up code.

**Why use `finally`?** For clean-up that must happen no matter what: closing a file, closing a database connection, logging "done".

In [ ]:
def convert(text):
    try:
        value = int(text)
    except ValueError:
        print(f"  except : '{text}' is not a number")
    else:
        print(f"  else   : converted to {value}")
    finally:
        print("  finally: always runs")

print("convert('7'):")
convert("7")
print("convert('seven'):")
convert("seven")

`finally` runs even when the `try` block **returns**:

In [ ]:
def check():
    try:
        return "returned from try"
    finally:
        print("finally ran before the function returned")

print(check())

---
## 5. `raise`: reporting errors from your own code

**What it is:** `raise` lets your code signal a problem. Use it when a function gets input it can't work with, so the caller finds out immediately instead of getting a silently wrong result.

**Behind the scenes:** `raise` creates the exception object and starts exactly the same "look for a handler" process as a built-in error.

**Syntax:** `raise ExceptionType("a helpful message")`

`isinstance(value, int)` (used below) returns `True` if `value` is an `int`.

In [ ]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError("age must be an integer")
    if age < 0:
        raise ValueError(f"age cannot be negative, got {age}")
    return age

print(set_age(25))

try:
    set_age(-5)
except ValueError as e:
    print("ValueError:", e)

try:
    set_age("25")
except TypeError as e:
    print("TypeError:", e)

**Re-raising:** a bare `raise` inside an `except` block sends the **same** exception on to the caller. Use it when you want to log something but can't fully handle the error yourself.

In [ ]:
def load_setting(text):
    try:
        return int(text)
    except ValueError:
        print("log: bad setting value, passing the error on")
        raise                     # re-raise the same exception

try:
    load_setting("abc")
except ValueError as e:
    print("caller handled it:", e)

---
## 6. Exceptions travel up through function calls

This is step 3 of "behind the scenes" in action. If a function doesn't handle an exception, the exception moves up to whichever function called it, and so on.

Here `float()` fails two levels down, and the error travels up `read_price` → `total_cost` → `checkout`, where it's finally handled. **Handle errors at the level that knows what to do about them.** Here that's `checkout`, which can tell the user.

In [ ]:
def read_price(text):
    return float(text)                         # may raise ValueError; not handled here

def total_cost(price_text, quantity):
    return read_price(price_text) * quantity   # not handled here either

def checkout(price_text, quantity):
    try:
        cost = total_cost(price_text, quantity)
    except ValueError:
        print(f"Invalid price: '{price_text}'")  # handled here, two levels up
    else:
        print(f"Total: {cost:.2f}")

checkout("99.5", 2)
checkout("abc", 2)

---
## 7. Custom exceptions

You can create your own exception type by making a class that **inherits** from `Exception`. Classes are covered in [04_oops.ipynb](04_oops.ipynb); for now, this two-line pattern is all you need.

**Why bother?** Callers can catch **your** specific error without accidentally catching unrelated ones, and the name explains what went wrong.

In [ ]:
class InsufficientBalanceError(Exception):
    pass

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientBalanceError(f"balance is {balance}, tried to withdraw {amount}")
    return balance - amount

try:
    print(withdraw(500, 200))
    print(withdraw(500, 900))
except InsufficientBalanceError as e:
    print("InsufficientBalanceError:", e)

---
## 8. Best practices

| Do | Don't |
|---|---|
| Catch specific exceptions: `except ValueError:` | Use a bare `except:` |
| Keep the `try` block small: only the line(s) that can fail | Wrap a whole function in one big `try` |
| Handle the error where you can do something useful | Catch and ignore it: `except: pass` |
| Use `finally` (or `with`, coming in file handling) for clean-up | Put clean-up after the `try` and hope it runs |
| Give `raise` a clear message | `raise ValueError()` with no message |

---
## 9. Practice

**1.** Write `safe_int(text, default=0)` that returns `int(text)`, or `default` if the conversion fails.

In [ ]:
# Exercise 1

**2.** Write `get_marks(marks, name)` that returns the student's mark from the dict `marks`. If the name is missing, print `"Student not found"` and return `None`. Use `try` / `except`, not `.get()`.

In [ ]:
# Exercise 2

**3.** Write `average(numbers)` that raises `ValueError("list is empty")` for an empty list. Call it with `[]` inside `try` / `except` and print the message.

In [ ]:
# Exercise 3

**4.** Ask the user for two numbers with `input()` and print the first divided by the second. Handle a non-number and division by zero with **separate** messages, use `else` to print the result, and use `finally` to print `"Done"`.

In [ ]:
# Exercise 4